# Author metadata: Semantic Scholar API vs OpenAlex

Which author metadata each source has, **checked with queries**, not from documentation:

- **Semantic Scholar:** every field is requested from the API; the API rejects fields that do not exist (`Unrecognized or unsupported fields`), so a field exists only if it is accepted. For a concept with no obvious field name (country, gender), the plausible names are all tried.
- **OpenAlex:** the column must exist in the schema of the local 2026-09 database (or of the snapshot's parquet author files, for `x_concepts`).

Then, on the same researchers in both sources (`factuality_author = True`, OpenAlex profile found through their papers), how often each field is filled.

In [1]:
import json
import sys
import time
import urllib.error
import urllib.request

import duckdb
import pandas as pd

sys.path.insert(0, "../../..")
from libs.utils.config import config_default

R = "../../../../results/summary"
N = 300
sample = pd.read_parquet(f"{R}/openalex_cascade/doi_match.parquet").sample(N, random_state=5)
ids = sample["researcher_id"].astype(str).tolist()
API = "https://api.semanticscholar.org/graph/v1/author/batch"


def ss_request(fields, ids):
    """(accepted, answer): accepted is False when the API says the field does not exist."""
    for attempt in range(30):
        try:
            req = urllib.request.Request(f"{API}?fields={fields}", data=json.dumps({"ids": ids}).encode(),
                                         headers={"Content-Type": "application/json"}, method="POST")
            return True, [a for a in json.load(urllib.request.urlopen(req, timeout=120)) if a]
        except urllib.error.HTTPError as e:
            if e.code == 400 and b"Unrecognized" in e.read():
                return False, []
            if e.code != 429:
                raise
            time.sleep(3 * (attempt + 1))

## 0. A complete answer from each source
Every field the Semantic Scholar API accepts for an author (and for their papers), requested at once for one researcher of the sample, and the full row of the same person in the OpenAlex database. Names are hidden.

In [2]:
AUTHOR_FIELDS = ["authorId", "externalIds", "url", "name", "affiliations", "homepage", "paperCount", "citationCount", "hIndex"]
PAPER_FIELDS = ["paperId", "corpusId", "externalIds", "url", "title", "abstract", "venue", "publicationVenue", "year",
                "referenceCount", "citationCount", "influentialCitationCount", "isOpenAccess", "openAccessPdf",
                "fieldsOfStudy", "s2FieldsOfStudy", "publicationTypes", "publicationDate", "journal", "citationStyles", "authors"]
# (these are all the fields the API accepted when each was tried on its own; others such as country or gender are rejected)

one = ids[0]
fields = ",".join(AUTHOR_FIELDS + [f"papers.{f}" for f in PAPER_FIELDS])
ok, answer = ss_request(fields, [one])
author = answer[0]


def shape(v):
    """The value's type and, for lists, their length; names and titles hidden."""
    if isinstance(v, list):
        return f"list of {len(v)}"
    if isinstance(v, dict):
        return "dict: " + ", ".join(f"{k}" for k in v)
    return type(v).__name__ if v is not None else "null"

print("Semantic Scholar, author:")
for k in AUTHOR_FIELDS:
    v = author.get(k)
    print(f"  {k:15s} {shape(v):35s}", "" if k in ("name", "url", "authorId") else (v if not isinstance(v, (list, dict)) else ""))
paper = (author.get("papers") or [{}])[0]
print("\nSemantic Scholar, one of their papers:")
for k in PAPER_FIELDS:
    v = paper.get(k)
    print(f"  {k:25s} {shape(v)}")
authors_of_paper = paper.get("authors") or []
print("\n  keys of each coauthor in papers.authors:", sorted({k for a in authors_of_paper for k in a}))

Semantic Scholar, author:
  authorId        str                                 
  externalIds     dict: DBLP                          
  url             str                                 
  name            str                                 
  affiliations    list of 0                           
  homepage        null                                None
  paperCount      int                                 434
  citationCount   int                                 20951
  hIndex          int                                 68

Semantic Scholar, one of their papers:
  paperId                   str
  corpusId                  int
  externalIds               dict: PubMedCentral, DOI, CorpusId, PubMed
  url                       str
  title                     str
  abstract                  str
  venue                     str
  publicationVenue          dict: id, name, type, alternate_names, issn, url, alternate_urls
  year                      int
  referenceCount            int
  cit

In [3]:
con = duckdb.connect()
con.execute(f"ATTACH '{config_default('oa_duckdb')}' AS oa (READ_ONLY)")
oa_id = int(sample.loc[sample["researcher_id"].astype(str) == one, "oa_id"].iloc[0].replace("https://openalex.org/A", ""))
row = con.execute(f"SELECT * FROM oa.authors WHERE id = {oa_id}").df().iloc[0]
con.close()
print("OpenAlex, author (database row):")
for k, v in row.items():
    if hasattr(v, "tolist"):
        v = v.tolist()
    print(f"  {k:28s} {shape(v)}")
first_inst = (row["last_known_institutions"].tolist() or [{}])[0]
print("\n  keys of an institution:", sorted(first_inst))

OpenAlex, author (database row):
  id                           int
  display_name                 str
  display_name_alternatives    list of 7
  orcid                        str
  works_count                  int
  cited_by_count               int
  summary_stats                dict: 2yr_mean_citedness, h_index, i10_index
  ids                          dict: orcid, observed_orcids
  affiliations                 list of 52
  last_known_institutions      list of 5
  topics                       list of 5
  counts_by_year               list of 34
  created_date                 Timestamp
  updated_date                 Timestamp

  keys of an institution: ['country_code', 'display_name', 'id', 'lineage', 'ror', 'type']


## 1. Semantic Scholar: which fields the API accepts, and how often they are filled

In [4]:
def paper_values(authors, key):
    return [p.get(key) for a in authors for p in a.get("papers") or []]

# what → (field names to try in the API, how to read the value from one author answer)
SS = {
    "name / alternative names": (["name", "aliases"], lambda a, f: a.get(f)),
    "ORCID": (["externalIds"], lambda a, f: (a.get("externalIds") or {}).get("ORCID")),
    "publications": (["paperCount"], lambda a, f: a.get(f)),
    "citations": (["citationCount"], lambda a, f: a.get(f)),
    "h-index": (["hIndex"], lambda a, f: a.get(f)),
    "i10-index": (["i10Index", "i10index"], lambda a, f: a.get(f)),
    "2-year mean citedness": (["meanCitedness", "twoYearMeanCitedness", "summaryStats"], lambda a, f: a.get(f)),
    "first / last publication year": (["firstYear", "papers.year"], None),
    "field": (["fieldsOfStudy", "papers.fieldsOfStudy"], None),
    "institution": (["affiliations"], lambda a, f: a.get(f)),
    "country": (["country", "countryCode", "location", "affiliations.country"], lambda a, f: a.get(f)),
    "coauthors": (["papers.authors"], None),
    "papers with DOI": (["papers.externalIds"], None),
    "citations per paper (for i10 / e-index)": (["papers.citationCount"], None),
    "gender": (["gender", "sex"], lambda a, f: a.get(f)),
}
PAPER_KEY = {"papers.year": "year", "papers.fieldsOfStudy": "fieldsOfStudy", "papers.authors": "authors",
             "papers.externalIds": "externalIds", "papers.citationCount": "citationCount"}

rows = []
for what, (names, read) in SS.items():
    found = None
    for f in names:
        batch = ids[:100] if f.startswith("papers.") else ids
        accepted, answer = ss_request(f, batch)
        time.sleep(2)
        if not accepted:
            continue
        if f.startswith("papers."):
            vals = paper_values(answer, PAPER_KEY[f])
            if f == "papers.externalIds":
                vals = [(v or {}).get("DOI") for v in vals]
            filled, unit = (100 * sum(bool(v) for v in vals) / len(vals) if vals else 0), "% of papers"
        else:
            vals = [read(a, f) for a in answer]
            filled, unit = (100 * sum(v not in (None, [], "") for v in vals) / len(vals) if vals else 0), "% of researchers"
        found = (f, round(filled, 1), unit)
        break
    rows.append({"what": what, "SS field": found[0] if found else "none of " + ", ".join(names),
                 "SS has it": int(found is not None and found[1] > 0), "SS filled": found[1] if found else 0.0,
                 "SS unit": found[2] if found else ""})
ss_table = pd.DataFrame(rows)
ss_table

,what,SS field,SS has it,SS filled,SS unit
0,name / alternative names,name,1,100.0,% of researchers
1,ORCID,externalIds,1,1.0,% of researchers
2,publications,paperCount,1,100.0,% of researchers
3,citations,citationCount,1,100.0,% of researchers
4,h-index,hIndex,1,100.0,% of researchers
5,i10-index,"none of i10Index, i10index",0,0.0,
6,2-year mean citedness,"none of meanCitedness, twoYearMeanCitedness, s...",0,0.0,
7,first / last publication year,papers.year,1,100.0,% of papers
8,field,papers.fieldsOfStudy,1,89.9,% of papers
9,institution,affiliations,1,1.3,% of researchers


## 2. OpenAlex: which columns exist, and how often they are filled

In [5]:
con = duckdb.connect()
con.execute(f"ATTACH '{config_default('oa_duckdb')}' AS oa (READ_ONLY)")
author_cols = {r[0]: r[1] for r in con.execute("DESCRIBE oa.authors").fetchall()}
work_cols = {r[0]: r[1] for r in con.execute("DESCRIBE oa.works").fetchall()}
parquet = "/code/openalex/oa_data/data/parquet/authors/updated_date=*/part_*.parquet"
parquet_cols = {r[0] for r in duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{parquet}')").fetchall()}

con.register("s", pd.DataFrame({"id": sample["oa_id"].str.replace("https://openalex.org/A", "").astype("int64")}))
oa = con.execute("SELECT a.* FROM oa.authors a JOIN s USING (id)").df()
con.close()

def filled(col, f=lambda v: v is not None and not (hasattr(v, "__len__") and len(v) == 0)):
    return round(100 * oa[col].map(f).mean(), 1)

OA = {
    "name / alternative names": ("authors.display_name_alternatives", lambda: filled("display_name_alternatives")),
    "ORCID": ("authors.orcid", lambda: filled("orcid")),
    "publications": ("authors.works_count", lambda: filled("works_count")),
    "citations": ("authors.cited_by_count", lambda: filled("cited_by_count")),
    "h-index": ("authors.summary_stats.h_index", lambda: filled("summary_stats", lambda v: v is not None and v.get("h_index") is not None)),
    "i10-index": ("authors.summary_stats.i10_index", lambda: filled("summary_stats", lambda v: v is not None and v.get("i10_index") is not None)),
    "2-year mean citedness": ("authors.summary_stats.2yr_mean_citedness",
                              lambda: filled("summary_stats", lambda v: v is not None and v.get("2yr_mean_citedness") is not None)),
    "first / last publication year": ("authors.counts_by_year", lambda: filled("counts_by_year")),
    "field": ("authors.topics", lambda: filled("topics")),
    "institution": ("authors.last_known_institutions", lambda: filled("last_known_institutions")),
    "country": ("authors.last_known_institutions.country_code",
                lambda: filled("last_known_institutions", lambda v: v is not None and any(i.get("country_code") for i in v))),
    "coauthors": ("works.authorships", None),
    "papers with DOI": ("works.doi", None),
    "citations per paper (for i10 / e-index)": ("works.cited_by_count", None),
    "gender": ("—", None),
}

rows = []
for what, (col, fill) in OA.items():
    table, _, name = col.partition(".")
    root = name.split(".")[0]
    exists = (table == "authors" and root in author_cols) or (table == "works" and root in work_cols)
    rows.append({"what": what, "OA column": col if exists else "none (gender, sex not in the schema)" if what == "gender" else col,
                 "OA has it": int(exists), "OA filled": fill() if exists and fill else (None if exists else 0.0),
                 "OA unit": "% of researchers" if exists and fill else ("in the works table" if exists else "")})
oa_table = pd.DataFrame(rows)
print("gender-like columns in authors:", [c for c in author_cols if "gender" in c or "sex" in c],
      "| in the parquet files:", [c for c in parquet_cols if "gender" in c or "sex" in c])
oa_table

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

gender-like columns in authors: [] | in the parquet files: []


,what,OA column,OA has it,OA filled,OA unit
0,name / alternative names,authors.display_name_alternatives,1,100.0,% of researchers
1,ORCID,authors.orcid,1,60.3,% of researchers
2,publications,authors.works_count,1,100.0,% of researchers
3,citations,authors.cited_by_count,1,100.0,% of researchers
4,h-index,authors.summary_stats.h_index,1,100.0,% of researchers
5,i10-index,authors.summary_stats.i10_index,1,100.0,% of researchers
6,2-year mean citedness,authors.summary_stats.2yr_mean_citedness,1,100.0,% of researchers
7,first / last publication year,authors.counts_by_year,1,100.0,% of researchers
8,field,authors.topics,1,100.0,% of researchers
9,institution,authors.last_known_institutions,1,83.7,% of researchers


## 3. Comparative table
1 = the source has it (checked above), 0 = it does not; *filled* = how often it comes filled on the sample (per researcher, or per paper).

In [ ]:
def pct(v, unit):
    if v is None or v != v:
        return unit or ""
    return f"{v:.0f}% of {unit.replace('% of ', '')}" if unit else ""

table = ss_table.merge(oa_table, on="what")
table = pd.DataFrame({
    "metadata": table["what"],
    "Semantic Scholar (API)": table["SS has it"],
    "SS field": [f if h else "—" for f, h in zip(table["SS field"], table["SS has it"])],
    "SS filled": [pct(v, u) if h else "—" for v, u, h in zip(table["SS filled"], table["SS unit"], table["SS has it"])],
    "OpenAlex (DB)": table["OA has it"],
    "OA field": [f if h else "—" for f, h in zip(table["OA column"], table["OA has it"])],
    "OA filled": [pct(v, u) if h else "—" for v, u, h in zip(table["OA filled"], table["OA unit"], table["OA has it"])],
})
table["only in"] = ["both" if s and o else "OpenAlex" if o else "Semantic Scholar" if s else "neither"
                    for s, o in zip(table["Semantic Scholar (API)"], table["OpenAlex (DB)"])]
table


,metadata,Semantic Scholar (API),SS field,SS filled,OpenAlex (DB),OA field,OA filled,only in
0,name / alternative names,1,name,100% of researchers,1,authors.display_name_alternatives,100% of researchers,both
1,ORCID,1,externalIds,1% of researchers,1,authors.orcid,60% of researchers,both
2,publications,1,paperCount,100% of researchers,1,authors.works_count,100% of researchers,both
3,citations,1,citationCount,100% of researchers,1,authors.cited_by_count,100% of researchers,both
4,h-index,1,hIndex,100% of researchers,1,authors.summary_stats.h_index,100% of researchers,both
5,i10-index,0,—,—,1,authors.summary_stats.i10_index,100% of researchers,OpenAlex
6,2-year mean citedness,0,—,—,1,authors.summary_stats.2yr_mean_citedness,100% of researchers,OpenAlex
7,first / last publication year,1,papers.year,100% of papers,1,authors.counts_by_year,100% of researchers,both
8,field,1,papers.fieldsOfStudy,90% of papers,1,authors.topics,100% of researchers,both
9,institution,1,affiliations,1% of researchers,1,authors.last_known_institutions,84% of researchers,both


## 4. Do the values agree?
Semantic Scholar and OpenAlex count different corpora, so the values are not expected to be equal; the correlation says whether they rank researchers the same way.

In [7]:
ok, answer = ss_request("paperCount,citationCount,hIndex", ids)
s = pd.DataFrame([{"researcher_id": int(a["authorId"]), "ss_papers": a.get("paperCount"),
                   "ss_citations": a.get("citationCount"), "ss_h": a.get("hIndex")} for a in answer])
s = s.merge(sample[["researcher_id", "oa_id"]], on="researcher_id")
s["id"] = s["oa_id"].str.replace("https://openalex.org/A", "").astype("int64")
o = oa.assign(oa_h=oa["summary_stats"].map(lambda x: (x or {}).get("h_index")))
both = s.merge(o[["id", "works_count", "cited_by_count", "oa_h"]], on="id")
pd.DataFrame({
    "median SS": [both["ss_papers"].median(), both["ss_citations"].median(), both["ss_h"].median()],
    "median OA": [both["works_count"].median(), both["cited_by_count"].median(), both["oa_h"].median()],
    "Spearman": [both[["ss_papers", "works_count"]].corr("spearman").iloc[0, 1],
                 both[["ss_citations", "cited_by_count"]].corr("spearman").iloc[0, 1],
                 both[["ss_h", "oa_h"]].corr("spearman").iloc[0, 1]],
}, index=["publications", "citations", "h-index"]).round(3)

,median SS,median OA,Spearman
publications,22.0,70.0,0.608
citations,238.0,910.5,0.663
h-index,6.0,15.0,0.634
